<a href="https://colab.research.google.com/github/Albert1616/ProjetoCAD-KMeans/blob/KmeansCUDA/KmeansCUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
%%writefile aluno.h

#ifndef ALUNO_H
#define ALUNO_H

typedef struct
{
    float media;
    float numeroFaltas;
    int cluster;
} Aluno;

#endif // ALUNO_H

Overwriting aluno.h


In [47]:
%%writefile dataset.h

#ifndef DATASET_H
#define DATASET_H
#include "aluno.h"

#ifdef __CUDACC__
__host__ char *retornarDadoPorIndice(char *linha, int index);
__host__ int carregarDataset(Aluno *alunos);
__host__ void normalizarAlunos(Aluno *alunos, int total);
#else
char *retornarDadoPorIndice(char *linha, int index);
int carregarDataset(Aluno *alunos);
void normalizarAlunos(Aluno *alunos, int total);
#endif

#endif // DATASET_H

Overwriting dataset.h


In [48]:
%%writefile kmeans.h

#ifndef KMEANS_H
#define KMEANS_H
#include "aluno.h"

typedef struct
{
    int k;
    int max_iter;
    int random_state;
    int totalAlunos;
    Aluno *centroids;
} KMeans;

#ifdef __CUDACC__
__global__ void initCentroids(KMeans *model, Aluno *alunos);
__global__ void assignClusters(KMeans *model, Aluno *alunos);
__global__ void updateCentroids(KMeans *model, Aluno *alunos);
__global__ void resetCentroids_kernel(Aluno *new_centroids, int *counts, int k);
__global__ void averageAndCheck_kernel(KMeans *model, Aluno *new_centroids, int *counts, int *d_convergiu);
__global__ void predict(KMeans *model, Aluno *novoAluno);
__host__ void fit(KMeans *model, Aluno *alunos);
__host__ float *methodElbow(KMeans *h_model, KMeans *d_model, Aluno *h_alunos, Aluno *d_alunos);
#endif

#endif // KMEANS_H

Overwriting kmeans.h


In [49]:
%%writefile kmeans.cu
#include <math.h>
#include <stdlib.h>
#include <stdio.h>
#include "aluno.h"
#include "kmeans.h"

#define MAX_K 10

// FUNÇÕES DEVICE / KERNELS
__device__ float distEuclidiana(Aluno *a, Aluno *b) {
    float diffMedia = a->media - b->media;
    float diffFaltas = a->numeroFaltas - b->numeroFaltas;
    return sqrtf(diffMedia * diffMedia + diffFaltas * diffFaltas);
}

// 1. Kernel para iniciar os centroides (Roda em 1 única thread para evitar duplicatas)
__global__ void initCentroids_kernel(KMeans *model, Aluno *alunos) {
    if (threadIdx.x == 0 && blockIdx.x == 0) {
        int listIndex[MAX_K];
        for (int i = 0; i < model->k; i++) listIndex[i] = -1;

        for (int i = 0; i < model->k; i++) {
            unsigned int index = (model->random_state + i * 1103515245U) % model->totalAlunos;            int duplicado = 0;
            for (int j = 0; j < i; j++) {
                if (listIndex[j] == index) { duplicado = 1; break; }
            }
            if (duplicado) { i--; continue; }
            listIndex[i] = index;
            model->centroids[i] = alunos[index];
        }
    }
}

// 2. Kernel para distribuir os alunos nos clusters
__global__ void assignClusters_kernel(KMeans *model, Aluno *alunos) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < model->totalAlunos) {
        float min_dist = 1e9f;
        int best_cluster = 0;

        for (int j = 0; j < model->k; j++) {
            float dist = distEuclidiana(&alunos[id], &model->centroids[j]);
            if (dist < min_dist) {
                min_dist = dist;
                best_cluster = j;
            }
        }
        alunos[id].cluster = best_cluster;
    }
}

// 3. Kernel para zerar os centroides temporários da iteração atual
__global__ void resetCentroids_kernel(Aluno *new_centroids, int *counts, int k) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < k) {
        new_centroids[id].media = 0.0f;
        new_centroids[id].numeroFaltas = 0.0f;
        counts[id] = 0;
    }
}

// 4. Kernel para somar as notas usando ATOMIC ADD (Evita Race Condition)
__global__ void updateCentroids_kernel(Aluno *alunos, Aluno *new_centroids, int *counts, int totalAlunos) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < totalAlunos) {
        int c = alunos[id].cluster;
        // atomicAdd enfileira o acesso na memória de hardware para evitar corrompimento
        atomicAdd(&new_centroids[c].media, alunos[id].media);
        atomicAdd(&new_centroids[c].numeroFaltas, alunos[id].numeroFaltas);
        atomicAdd(&counts[c], 1);
    }
}

// 5. Kernel para dividir pela média e verificar a convergência
__global__ void averageAndCheck_kernel(KMeans *model, Aluno *new_centroids, int *counts, int *d_convergiu) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < model->k) {
        if (counts[id] > 0) {
            new_centroids[id].media /= counts[id];
            new_centroids[id].numeroFaltas /= counts[id];
        }

        // Verifica a distância entre o centroide velho e o recém calculado
        float dist = distEuclidiana(&model->centroids[id], &new_centroids[id]);
        if (dist > 0.0001f) {
            *d_convergiu = 0;
        }

        // Salva os novos valores como definitivos para a próxima rodada
        model->centroids[id].media = new_centroids[id].media;
        model->centroids[id].numeroFaltas = new_centroids[id].numeroFaltas;
    }
}

__global__ void predict(KMeans *model, Aluno *novoAluno) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id == 0) {
        float min_dist = 1e9f;
        int best_cluster = 0;
        for (int i = 0; i < model->k; i++) {
            float dist = distEuclidiana(novoAluno, &model->centroids[i]);
            if (dist < min_dist) {
                min_dist = dist;
                best_cluster = i;
            }
        }
        novoAluno->cluster = best_cluster;
    }
}


// FUNÇÕES HOST (Rodam na CPU orquestrando a Placa de Vídeo)

__host__ void fit(KMeans *d_model, Aluno *d_alunos) {
    // 1. Trazemos o modelo cru (vazio) de volta para a CPU apenas para extrair as variáveis 'k', 'max_iter' e 'total'
    KMeans h_model;
    cudaMemcpy(&h_model, d_model, sizeof(KMeans), cudaMemcpyDeviceToHost);

    int k = h_model.k;
    int max_iter = h_model.max_iter;
    int totalAlunos = h_model.totalAlunos;

    // 2. Definimos a malha de execução CUDA
    int threads = 256;
    int blocosAlunos = (totalAlunos + threads - 1) / threads;
    int blocosK = (k + threads - 1) / threads;

    // 3. Alocamos buffers temporários na GPU necessários para o AtomicAdd
    Aluno *d_new_centroids;
    int *d_counts;
    int *d_convergiu;

    cudaMalloc(&d_new_centroids, k * sizeof(Aluno));
    cudaMalloc(&d_counts, k * sizeof(int));
    cudaMalloc(&d_convergiu, sizeof(int));

    // Inicializa os centroides
    initCentroids_kernel<<<1, 1>>>(d_model, d_alunos);

    // 4. O Loop Principal do K-Means orquestrado pela CPU
    for (int iter = 0; iter < max_iter; iter++) {

        // Zera os buffers temporários de soma
        resetCentroids_kernel<<<blocosK, threads>>>(d_new_centroids, d_counts, k);

        // Cada thread pega 1 aluno e calcula a distância para todos os centroides
        assignClusters_kernel<<<blocosAlunos, threads>>>(d_model, d_alunos);

        // Cada thread soma as notas do seu aluno no centroide respectivo (com atomicAdd)
        updateCentroids_kernel<<<blocosAlunos, threads>>>(d_alunos, d_new_centroids, d_counts, totalAlunos);

        // Assume na CPU que já convergiu (verdadeiro) e envia para a GPU julgar
        int h_convergiu = 1;
        cudaMemcpy(d_convergiu, &h_convergiu, sizeof(int), cudaMemcpyHostToDevice);

        // A GPU tira as médias e se algum centroide andar mais que 0.01, ela seta d_convergiu para 0
        averageAndCheck_kernel<<<blocosK, threads>>>(d_model, d_new_centroids, d_counts, d_convergiu);

        // Traz o resultado da convergência de volta pra CPU para decidir se interrompe o while
        cudaMemcpy(&h_convergiu, d_convergiu, sizeof(int), cudaMemcpyDeviceToHost);

        if (h_convergiu) {
            printf("\nTreinamento finalizado. Modelo convergiu na iteracao %d.\n", iter);
            break;
        }
    }

    // Limpa a memória temporária da GPU
    cudaFree(d_new_centroids);
    cudaFree(d_counts);
    cudaFree(d_convergiu);
}

Overwriting kmeans.cu


In [50]:
%%writefile dataset.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "aluno.h"
#include "dataset.h"

char *retornarDadoPorIndice(char *linha, int index)
{
    char copia[strlen(linha) + 1];
    strcpy(copia, linha);
    int contador = 0;

    char *dados = strtok(copia, ",");

    while (dados != NULL)
    {
        if (contador == index)
        {
            return strdup(dados);
        }
        dados = strtok(NULL, ",");
        contador++;
    }

    return NULL;
}

int carregarDataset(Aluno *alunos)
{
    FILE *file = fopen("Data/Student_performance_data.csv", "r");

    if (file == NULL)
    {
        printf("Erro ao abrir o arquivo.\n");
        return 0;
    }

    char linha[4000];
    int index = 0;

    while (fgets(linha, sizeof(linha), file) != NULL)
    {
        if (index == 0)
        {
            index++;
            continue;
        }

        char copia[4000];
        strcpy(copia, linha);

        char *colunas[15];
        colunas[0] = strtok(copia, ",");
        for (int i = 1; i < 15; i++)
            colunas[i] = strtok(NULL, ",");

        Aluno aluno;
        aluno.numeroFaltas = atof(colunas[6]);
        aluno.media = atof(colunas[13]);
        aluno.cluster = -1;

        alunos[index - 1] = aluno;
        index++;
    }

    fclose(file);
    return index - 1;
}

void normalizarAlunos(Aluno *alunos, int total)
{
    float minMedia = alunos[0].media, maxMedia = alunos[0].media;
    float minFaltas = alunos[0].numeroFaltas, maxFaltas = alunos[0].numeroFaltas;

    for (int i = 1; i < total; i++)
    {
        if (alunos[i].media < minMedia)
            minMedia = alunos[i].media;
        if (alunos[i].media > maxMedia)
            maxMedia = alunos[i].media;

        if (alunos[i].numeroFaltas < minFaltas)
            minFaltas = alunos[i].numeroFaltas;
        if (alunos[i].numeroFaltas > maxFaltas)
            maxFaltas = alunos[i].numeroFaltas;
    }

    for (int i = 0; i < total; i++)
    {
        alunos[i].media = (alunos[i].media - minMedia) / (maxMedia - minMedia);
        alunos[i].numeroFaltas = (alunos[i].numeroFaltas - minFaltas) / (maxFaltas - minFaltas);
    }
}

Overwriting dataset.cu


In [53]:
%%writefile main.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime_api.h>
#include "aluno.h"
#include "dataset.h"
#include "kmeans.h"

#define NUM_CLUSTERS 3
#define MAX_ITERATIONS 10000

void exportarResultados(Aluno *alunos, int total)
{
    FILE *file = fopen("resultados.csv", "w");
    fprintf(file, "numeroFaltas,media,cluster\n");

    for (int i = 0; i < total; i++)
        fprintf(file, "%.2f,%.2f,%d\n",
                alunos[i].numeroFaltas,
                alunos[i].media,
                alunos[i].cluster);

    fclose(file);
}

int main()
{
    Aluno *h_alunos = (Aluno *)malloc(3000 * sizeof(Aluno));
    int numeroAlunos = carregarDataset(h_alunos);
    normalizarAlunos(h_alunos, numeroAlunos);

    // aloca e copia alunos para GPU
    Aluno *d_alunos;
    cudaMalloc(&d_alunos, numeroAlunos * sizeof(Aluno));
    cudaMemcpy(d_alunos, h_alunos, numeroAlunos * sizeof(Aluno), cudaMemcpyHostToDevice);

    // configura modelo na CPU
    KMeans h_kmeans;
    h_kmeans.k = NUM_CLUSTERS;
    h_kmeans.max_iter = MAX_ITERATIONS;
    h_kmeans.random_state = 42;
    h_kmeans.totalAlunos = numeroAlunos;
    h_kmeans.centroids = (Aluno *)malloc(h_kmeans.k * sizeof(Aluno));

    // aloca centroids na GPU e atualiza ponteiro no modelo
    Aluno *d_centroids;
    cudaMalloc(&d_centroids, h_kmeans.k * sizeof(Aluno));

    // copia modelo para GPU
    KMeans *d_kmeans;
    cudaMalloc(&d_kmeans, sizeof(KMeans));
    cudaMemcpy(d_kmeans, &h_kmeans, sizeof(KMeans), cudaMemcpyHostToDevice);

    // corrige ponteiro de centroids dentro do d_kmeans
    cudaMemcpy(&(d_kmeans->centroids), &d_centroids, sizeof(Aluno *), cudaMemcpyHostToDevice);

    // treinamento: A CPU chama a função fit para orquestrar os kernels
    fit(d_kmeans, d_alunos);

    // traz resultados para CPU
    cudaMemcpy(h_alunos, d_alunos, numeroAlunos * sizeof(Aluno), cudaMemcpyDeviceToHost);
    cudaMemcpy(h_kmeans.centroids, d_centroids, h_kmeans.k * sizeof(Aluno), cudaMemcpyDeviceToHost);

    exportarResultados(h_alunos, numeroAlunos);

    for (int i = 0; i < h_kmeans.k; i++)
        printf("Centroid %d - Media: %.2f, Numero de Faltas: %.2f\n", i,
               h_kmeans.centroids[i].media, h_kmeans.centroids[i].numeroFaltas);

    // predição
    Aluno novoAluno;
    novoAluno.media = 0.95;
    novoAluno.numeroFaltas = 0.1;

    Aluno *d_novoAluno;
    cudaMalloc(&d_novoAluno, sizeof(Aluno));
    cudaMemcpy(d_novoAluno, &novoAluno, sizeof(Aluno), cudaMemcpyHostToDevice);
    predict<<<1, 1>>>(d_kmeans, d_novoAluno);
    cudaDeviceSynchronize();
    cudaMemcpy(&novoAluno, d_novoAluno, sizeof(Aluno), cudaMemcpyDeviceToHost);

    printf("Novo aluno - Media: %.2f, Numero de Faltas: %.2f, Cluster: %d\n",
           novoAluno.media, novoAluno.numeroFaltas, novoAluno.cluster);

    // libera memória
    free(h_kmeans.centroids);
    free(h_alunos);
    cudaFree(d_alunos);
    cudaFree(d_kmeans);
    cudaFree(d_centroids);
    cudaFree(d_novoAluno);

    return 0;
}

Overwriting main.cu


In [54]:
!nvcc kmeans.cu dataset.cu main.cu -o kmeans
!./kmeans

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

Treinamento finalizado. Modelo convergiu na iteracao 14.
Centroid 0 - Media: 0.73, Numero de Faltas: 0.16
Centroid 1 - Media: 0.48, Numero de Faltas: 0.51
Centroid 2 - Media: 0.22, Numero de Faltas: 0.84
Novo aluno - Media: 0.95, Numero de Faltas: 0.10, Cluster: 0
